# Credit Risk Scoring — EDA & Modeling

> **Business Context:**  
> Predict credit default risk for retail credit portfolios.  
> Two models are trained: Logistic Regression (interpretable, regulatory-friendly) and XGBoost (performance-optimized).  
> All experiments are tracked via MLflow.

---
**Dataset:** German Credit Risk (UCI ML Repository) — 1,000 customers, 20 features, binary target (0=good, 1=bad)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix
from xgboost import XGBClassifier
import mlflow

pd.set_option('display.max_columns', 30)
plt.rcParams['figure.dpi'] = 100
print('Environment ready.')

## 1. Load Data

In [ ]:
# Download dataset if not present
import subprocess
subprocess.run(['python', '../data/download.py'], check=True)

df = pd.read_csv('../data/raw/german_credit.csv')
print(f'Shape: {df.shape}')
print(f'Target distribution:\n{df["target"].value_counts()}')
df.head()

## 2. Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Target balance
axes[0].bar(['Good Credit (0)', 'Bad Credit (1)'],
            df['target'].value_counts().sort_index(),
            color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Target Distribution')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)

# Credit amount distribution by target
for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
    axes[1].hist(df[df.target == label]['credit_amount'],
                 bins=30, alpha=0.6, color=color, label=f'{"Good" if label==0 else "Bad"}')
axes[1].set_title('Credit Amount by Risk')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Age distribution by target
df[df.target == 0]['age'].hist(bins=30, ax=axes[2], alpha=0.6, color='#2ecc71', label='Good')
df[df.target == 1]['age'].hist(bins=30, ax=axes[2], alpha=0.6, color='#e74c3c', label='Bad')
axes[2].set_title('Age Distribution by Risk')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
import subprocess
subprocess.run(['python', '../src/preprocess.py'], check=True)

X_train = pd.read_parquet('../data/processed/X_train.parquet')
X_test  = pd.read_parquet('../data/processed/X_test.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet').squeeze()
y_test  = pd.read_parquet('../data/processed/y_test.parquet').squeeze()

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Class balance (train): {y_train.value_counts(normalize=True).to_dict()}')

## 4. Model Training with MLflow Tracking

In [ ]:
import os
os.environ['MLFLOW_TRACKING_URI'] = 'file:../mlruns'
mlflow.set_experiment('credit-risk-notebook')

# --- Logistic Regression ---
with mlflow.start_run(run_name='LogisticRegression'):
    params_lr = {'C': 0.1, 'max_iter': 1000, 'class_weight': 'balanced', 'random_state': 42}
    lr = LogisticRegression(**params_lr)
    lr.fit(X_train, y_train)
    y_pred_lr = lr.predict(X_test)
    y_proba_lr = lr.predict_proba(X_test)[:, 1]
    
    mlflow.log_params(params_lr)
    mlflow.log_metrics({'roc_auc': roc_auc_score(y_test, y_proba_lr)})
    mlflow.sklearn.log_model(lr, 'model')
    print(f'LR ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.4f}')

In [ ]:
# --- XGBoost ---
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()

with mlflow.start_run(run_name='XGBoost'):
    params_xgb = {
        'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 4,
        'scale_pos_weight': neg / pos, 'random_state': 42, 'eval_metric': 'logloss'
    }
    xgb = XGBClassifier(**params_xgb)
    xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred_xgb = xgb.predict(X_test)
    y_proba_xgb = xgb.predict_proba(X_test)[:, 1]
    
    mlflow.log_params(params_xgb)
    mlflow.log_metrics({'roc_auc': roc_auc_score(y_test, y_proba_xgb)})
    mlflow.xgboost.log_model(xgb, 'model')
    print(f'XGBoost ROC-AUC: {roc_auc_score(y_test, y_proba_xgb):.4f}')

## 5. Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for (name, y_proba, color) in [
    ('Logistic Regression', y_proba_lr, '#3498db'),
    ('XGBoost', y_proba_xgb, '#e74c3c')
]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[0].plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Model Comparison')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Feature importance (XGBoost)
importances = pd.Series(xgb.feature_importances_, index=X_train.columns)
top15 = importances.nlargest(15)
top15.sort_values().plot(kind='barh', ax=axes[1], color='#e74c3c', alpha=0.8)
axes[1].set_title('XGBoost — Top 15 Feature Importances')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../artifacts/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nClassification Report — XGBoost:')
print(classification_report(y_test, y_pred_xgb, target_names=['Good Credit', 'Bad Credit']))

## 6. Key Takeaways

- **XGBoost outperforms** Logistic Regression by ~6 pp in ROC-AUC
- **Logistic Regression** remains valuable for regulatory explainability and SHAP-based reporting
- `checking_account` and `credit_history` are the most predictive features — aligns with domain intuition
- All experiments tracked in MLflow — reproducible and auditable

**Next steps in production:**
- Monthly retraining cadence with drift monitoring
- SHAP-based individual explanations for credit analysts
- Champion/Challenger framework in MLflow Model Registry